# SP³+SOC Tight-Binding Model for Halide Perovskites

This notebook demonstrates a complete tight-binding (TB) study of **3D bulk and 2D Ruddlesden-Popper halide perovskites** (CsPbI₃/MAPbI₃ family), covering:

1. The 26-parameter SP³+SOC Hamiltonian
2. Band gap comparison against experiment and DFT
3. Effective mass tensor (in-plane vs out-of-plane)
4. Optical absorption spectra (3D bulk + 2D slabs n=1–5)
5. Exciton binding energy via Wannier-Mott model
6. Model fit quality summary
7. *(Phase 2 — run separately)* ML surrogate + Bayesian optimization
8. **Quantum VQE** — 2-qubit VQE on the near-gap TB Hamiltonian subspace

**Backend:** Fortran 90 Hamiltonian compiled to a shared library, called from Python via `ctypes`. No external electronic structure package required.

**Reference:** Boyer-Richard et al., *J. Phys. Chem. Lett.* **7**, 3833 (2016).

## 0. Setup

Before running this notebook:
```bash
cd src/fortran
gfortran -O3 -shared -fPIC -o libhamiltonian_3rd.so ham2d_general.f90 ham2d_3rd.f90 -llapack -lblas
cp libhamiltonian_3rd.so ../python/
cd ../..
pip install numpy matplotlib joblib scipy
```

In [ ]:
import sys, os

# Add src/python to path so we can import ctypesroutines
REPO_ROOT = os.path.abspath('')
SRC_PY    = os.path.join(REPO_ROOT, 'src', 'python')
sys.path.insert(0, SRC_PY)
os.chdir(SRC_PY)   # ctypesroutines.py loads libhamiltonian_3rd.so from cwd

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

from ctypesroutines import (
    hamiltonian_cubic3D_fortran,
    hamiltonian_cubic2D_apical_fortran,
    diagonalize_hamiltonian_fortran,
    eigenvalues_only_fortran,
)
print('Fortran backend loaded successfully.')

## 1. The 26-Parameter SP³+SOC Hamiltonian

The model uses an sp³ basis (s + px + py + pz on each atom) with spin-orbit coupling (SOC).

| Group | Parameters | Count |
|---|---|---|
| On-site energies | Es0, Ep0 (Pb) + Es1, Ep1 (I) | 4 |
| Nearest-neighbour Pb-I | Vss, Vsp, Vps, Vppsig, Vpppi | 5 |
| SOC | δSO0 (Pb), δSO1 (I) | 2 |
| 2nd-neighbour I-I (in-plane) | Vss_II, Vsp_II, Vppsig_II, Vpppi_II | 4 |
| 2nd-neighbour I-I (out-of-plane) | Vss_IIz, Vsp_IIz, Vppsig_IIz, Vpppi_IIz | 4 |
| 2nd-neighbour Pb-Pb | Vss_PP, Vsp_PP, Vppsig_PP, Vpppi_PP | 4 |
| Surface corrections (2D only) | Es1_ap, Ep1_ap, Ep0_surf | 3 |

The **3D bulk** Hamiltonian uses the first 19 parameters (no IIz, no surface corrections).  
The **2D slab** Hamiltonian uses all 26, applying `Ep0_surf` to the outermost Pb layers.

In [ ]:
PARAMS = np.array([
    -2.75, +3.42, -9.42, +2.58, +1.95, +0.74, -3.64, -1.86, -0.90, +1.30, +0.90,
    -0.19, -0.59, +0.09, +0.27,
    -0.49, +0.01, +0.27, +0.20,
     0.0,   0.0,   0.0,   0.0,
    -0.16,  3.60,  3.52,
])

(Es0, Ep0, Es1, Ep1, Vss, Vsp, Vps, Vppsig, Vpppi,
 deltaSO0, deltaSO1,
 Vss_II, Vsp_II, Vppsig_II, Vpppi_II,
 Vss_IIz, Vsp_IIz, Vppsig_IIz, Vpppi_IIz,
 Vss_PbPb, Vsp_PbPb, Vppsig_PbPb, Vpppi_PbPb,
 Es1_ap, Ep1_ap, Ep0_surf) = PARAMS

P3D = [Es0, Ep0, Es1, Ep1, Vss, Vsp, Vps, Vppsig, Vpppi,
       deltaSO0, deltaSO1,
       Vss_II, Vsp_II, Vppsig_II, Vpppi_II,
       Vss_PbPb, Vsp_PbPb, Vppsig_PbPb, Vpppi_PbPb]

A_LAT  = 6.28    # lattice constant [Å]
IVB_3D = 25      # top valence band index (0-based, 32x32 matrix)
ICB_3D = 26      # bottom conduction band index

# 2D slab: IVB, ICB, matrix size for n=1..5
SLABS = {1:(33,34,40), 2:(59,60,72), 3:(85,86,104), 4:(111,112,136), 5:(137,138,168)}

print(f'Parameters loaded: {len(PARAMS)} total (19 bulk + 7 2D-specific)')
print(f'Lattice constant a = {A_LAT} Å')
print(f'Surface correction Ep0_surf − Ep0 = {Ep0_surf - Ep0:.3f} eV')

## 2. Hamiltonian at a Single k-Point

Let's build the Hamiltonian at the **R-point** — the fundamental gap location for 3D bulk.
This calls Fortran directly via `ctypes` and returns a 32×32 complex Hermitian matrix.

In [ ]:
kR = np.array([np.pi/A_LAT, np.pi/A_LAT, np.pi/A_LAT])
kM = np.array([np.pi/A_LAT, np.pi/A_LAT, 0.0])

H_3D = hamiltonian_cubic3D_fortran(kR[0], kR[1], kR[2], *P3D)
print(f'3D Hamiltonian at R: shape = {H_3D.shape}, dtype = {H_3D.dtype}')
print(f'Hermitian check  ||H - H†||_max = {np.max(np.abs(H_3D - H_3D.conj().T)):.2e}')

# Diagonalize
eigvals, eigvecs = diagonalize_hamiltonian_fortran(H_3D)
gap_3d = eigvals[ICB_3D] - eigvals[IVB_3D]
print(f'\n3D bulk band gap at R = {gap_3d:.4f} eV   (experiment: 1.650 eV)')
print(f'Error = {(gap_3d - 1.650)*1000:.1f} meV')

In [ ]:
# 2D slab gaps at M-point
EXP_GAPS = {1: 3.016, 2: 2.432, 3: 2.206, 4: 2.078, 5: 1.971}

print(f'{"n":>4}  {"TB gap (eV)":>12}  {"Exp. (eV)":>10}  {"Error (meV)":>12}')
print('-' * 44)

for n in range(1, 6):
    ivb, icb, _ = SLABS[n]
    H_2D = hamiltonian_cubic2D_apical_fortran(
        n, kM[0], kM[1], kM[2],
        Es0, Ep0, Es1, Ep1, Vss, Vsp, Vps, Vppsig, Vpppi,
        deltaSO0, deltaSO1,
        Vss_II, Vsp_II, Vppsig_II, Vpppi_II,
        Vss_IIz, Vsp_IIz, Vppsig_IIz, Vpppi_IIz,
        Vss_PbPb, Vsp_PbPb, Vppsig_PbPb, Vpppi_PbPb,
        Es1_ap, Ep1_ap, Ep0_surf)
    ev = eigenvalues_only_fortran(H_2D)
    gap = ev[icb] - ev[ivb]
    err = (gap - EXP_GAPS[n]) * 1000
    print(f'  {n:>2}  {gap:>12.4f}  {EXP_GAPS[n]:>10.3f}  {err:>+12.1f}')

## 3. Band Structure

Band structures computed along:
- **3D bulk:** M → R → Γ → X → M
- **2D slabs:** Γ → M → X → Γ

VBM is shifted to 0 eV. Red = VBM band, green = CBM band.

In [ ]:
Image(os.path.join(REPO_ROOT, 'results', 'figures', 'bands_3D.png'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, n in zip(axes, [1, 2, 3]):
    img = plt.imread(os.path.join(REPO_ROOT, 'results', 'figures', f'bands_2D_n{n}.png'))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'2D slab n={n}', fontsize=12)
plt.suptitle('Band structures for 2D Ruddlesden-Popper slabs', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Effective Mass Tensor

Effective masses are computed by finite differences of the eigenvalues:
$$\frac{1}{m^*} = \frac{1}{\hbar^2} \frac{d^2 E}{dk^2}$$

Key result: **2D slabs have m_z = ∞** (carriers are quantum-confined in the stacking direction).
As n → ∞, the in-plane masses converge to the 3D bulk value at R.

In [ ]:
Image(os.path.join(REPO_ROOT, 'results', 'figures', 'effective_mass_tensor.png'))

## 5. Optical Absorption Spectra

The absorption coefficient follows Boyer-Richard et al.:

$$\alpha(\omega) = \frac{\pi e^2}{\varepsilon_0 \hbar c} \cdot \frac{10^8}{a^3 n_r \omega N_k}
\sum_{cv,\mathbf{k}} \left|\left\langle v,\mathbf{k}\left|\frac{\partial H}{\partial k_x}\right|c,\mathbf{k}\right\rangle\right|^2
G(E_c - E_v - \omega,\, \eta)$$

The **momentum matrix elements** $\langle v|\partial H/\partial k_x|c\rangle$ are computed by central finite differences of $H(\mathbf{k})$, requiring no additional Fortran subroutine.

Settings: η = 0.05 eV broadening, 30² k-grid (2D), 20³ k-grid (3D), x-polarisation.

In [ ]:
# Load pre-computed absorption data
DATA_DIR = os.path.join(REPO_ROOT, 'results', 'data')

e3d, a3d = np.loadtxt(os.path.join(DATA_DIR, 'absorption_3D_bulk.dat')).T

slabs_data = {}
for n in range(1, 6):
    e, a = np.loadtxt(os.path.join(DATA_DIR, f'absorption_2D_n{n}.dat')).T
    slabs_data[n] = (e, a)

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(e3d, a3d, color='black', lw=2.0, ls='--', label='3D bulk', zorder=10)
for i, n in enumerate(range(1, 6)):
    e, a = slabs_data[n]
    ax.plot(e, a, color=COLORS[i], lw=1.5, label=f'2D n={n}')

ax.set_xlabel('Photon energy (eV)', fontsize=13)
ax.set_ylabel(r'$\alpha$ (cm$^{-1}$)', fontsize=13)
ax.set_title('Optical absorption: 3D bulk and 2D Ruddlesden-Popper slabs\nSP³ TB+SOC, x-polarisation, η = 0.05 eV', fontsize=11)
ax.set_xlim(1.4, 4.5)
ax.set_yscale('log')
ax.set_ylim(bottom=1)
ax.legend(fontsize=11, framealpha=0.85)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

# Print onset energies
print('\nAbsorption onset (first energy where α > 100 cm⁻¹):')
for n in range(1, 6):
    e, a = slabs_data[n]
    mask = a > 100
    onset = e[mask][0] if mask.any() else float('nan')
    print(f'  n={n}: {onset:.3f} eV')
mask = a3d > 100
print(f'  3D:  {e3d[mask][0]:.3f} eV')

## 6. Exciton Binding Energy — Wannier-Mott Model

The optical gap is lower than the electronic gap due to Coulomb attraction between
the photo-excited electron and hole:

$$E_b = R_y \cdot \frac{\mu}{m_0} \cdot \frac{1}{\varepsilon_{\rm eff}^2}$$

where $R_y = 13.606$ eV and the slab-averaged dielectric constant is:

$$\varepsilon_{\rm eff}(n) = \frac{n \cdot \varepsilon_{\rm well} + 2\, \varepsilon_{\rm barrier}}{n + 2}$$

This gives a **natural 1/n scaling** of the binding energy — the largest $E_b$ at n=1 (most 2D)
and exact recovery of the bulk value as n → ∞, with no empirical n-dependent prefactor.

In [ ]:
Image(os.path.join(REPO_ROOT, 'results', 'figures', 'exciton_binding.png'))

In [ ]:
# Reproduce the Wannier-Mott calculation from scratch
RYD         = 13.6056   # eV
eps_well    = 5.75
eps_barrier = 1.0

# Experimental reference (Blancon et al., Nat. Commun. 2018)
EXP_EG_ELEC = np.array([3.016, 2.432, 2.206, 2.078, 1.971])
EXP_EG_OPT  = np.array([2.57,  2.18,  2.04,  1.95,  1.89])
EXP_EB      = EXP_EG_ELEC - EXP_EG_OPT

n_vals   = np.arange(1, 6)
eps_eff  = (n_vals * eps_well + 2 * eps_barrier) / (n_vals + 2)

# Reduced masses from TB model (pre-computed values)
mu_tb = np.array([0.221, 0.217, 0.201, 0.196, 0.186])
eb_model = RYD * mu_tb / eps_eff**2

print(f'{"n":>3}  {"ε_eff":>6}  {"μ (m₀)":>8}  {"Eb model (meV)":>15}  {"Eb exp (meV)":>13}')
print('-' * 52)
for i, n in enumerate(n_vals):
    print(f'  {n:>1}  {eps_eff[i]:>6.2f}  {mu_tb[i]:>8.3f}  {eb_model[i]*1000:>15.1f}  {EXP_EB[i]*1000:>13.1f}')

# Plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(1/n_vals, eb_model*1000, 'o-', color='black', lw=2, ms=8, label='TB Wannier-Mott')
ax.plot(1/n_vals, EXP_EB*1000,   's--', color='tomato', lw=1.8, ms=7, label='Experiment')
ax.set_xlabel('1/n  (bulk at 0)', fontsize=12)
ax.set_ylabel('Exciton binding energy (meV)', fontsize=11)
ax.set_title('Quantum-confinement enhancement of Eᵦ', fontsize=12)
ax.set_xticks([0, 1/5, 1/4, 1/3, 1/2, 1])
ax.set_xticklabels(['bulk', '1/5', '1/4', '1/3', '1/2', 'n=1'])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Model Fit Quality

Summary of fit quality across all observables:

In [ ]:
Image(os.path.join(REPO_ROOT, 'results', 'figures', 'comparison_table.png'))

In [ ]:
Image(os.path.join(REPO_ROOT, 'results', 'figures', 'convergence_plot.png'))

## Summary

| Observable | Agreement |
|---|---|
| 3D bulk band gap at R | < 10 meV |
| 2D slab gaps n=1–5 at M | < 50 meV |
| Reduced masses μ | < 5% |
| Optical absorption onset | correct quantum-confinement trend |
| Exciton binding energy | correct 1/n scaling |

The 26 parameters are fitted once against DFT/Wannier band gaps using a genetic algorithm
followed by L-BFGS-B local refinement. The surface correction (`Ep0_surf`) generates the
correct 1/n band gap scaling from geometry alone — no empirical n-dependent term is needed.

---
*Full code and data: [github.com/bashalak/perovskite-tb-soc](https://github.com/bashalak/perovskite-tb-soc)*

---

## 8. Quantum Computing: Variational Quantum Eigensolver (VQE)

The TB Hamiltonian at a k-point is a Hermitian matrix — the same mathematical structure as a quantum system's energy operator. This makes it a natural candidate for quantum algorithms designed for electronic structure.

**Strategy:**
1. Diagonalise H at the R-point → extract 4 near-gap eigenstates (2 VB + 2 CB)
2. Evaluate H at a slightly shifted k-point, project onto the 4-state subspace → 4×4 H_eff
3. Encode H_eff as a 2-qubit observable and minimise ⟨ψ(θ)|H_eff|ψ(θ)⟩ using a quantum circuit

The full 32×32 Hamiltonian would require 32 qubits (2³² states ≈ 4 billion numbers). The 4×4 subspace needs only 2 qubits.

**Requires:** `pip install pennylane`

In [ ]:
try:
    import pennylane as qml
    import pennylane.numpy as pnp
    PENNYLANE_AVAILABLE = True
except ImportError:
    print("PennyLane not installed — skipping VQE cells.")
    print("Install with:  pip install pennylane")
    PENNYLANE_AVAILABLE = False

if PENNYLANE_AVAILABLE:
    DELTA_K  = 0.05 * np.pi / A_LAT   # 5% k-shift from R
    IVB, ICB = 25, 26
    sub_idx  = [IVB - 1, IVB, ICB, ICB + 1]

    # Step 1: R-point subspace basis
    kR = np.pi / A_LAT
    H_R = hamiltonian_cubic3D_fortran(kR, kR, kR, *P3D)
    evals_R, evecs_R = np.linalg.eigh(H_R)
    U4 = evecs_R[:, sub_idx]   # 32×4

    # Step 2: 4×4 projected H at shifted k
    H_k   = hamiltonian_cubic3D_fortran(kR + DELTA_K, kR, kR, *P3D)
    H_eff = U4.conj().T @ H_k @ U4

    E_exact = np.linalg.eigvalsh(H_eff)
    print(f"4×4 H_eff eigenvalues: {E_exact.round(4)} eV")
    print(f"Exact ground state:    {E_exact[0]:.6f} eV")
    print(f"\nH_eff Hermitian check: ||H - H†||_max = {np.max(np.abs(H_eff - H_eff.conj().T)):.2e}")

In [ ]:
if PENNYLANE_AVAILABLE:
    # Step 3: VQE — hardware-efficient 2-qubit ansatz, Adam optimiser
    N_VQE_STEPS = 300

    dev = qml.device("default.qubit", wires=2)

    @qml.qnode(dev)
    def vqe_circuit(params):
        qml.RY(params[0], wires=0);  qml.RY(params[1], wires=1)
        qml.CNOT(wires=[0, 1])
        qml.RY(params[2], wires=0);  qml.RY(params[3], wires=1)
        qml.CNOT(wires=[1, 0])
        qml.RY(params[4], wires=0);  qml.RY(params[5], wires=1)
        return qml.expval(qml.Hermitian(H_eff, wires=[0, 1]))

    rng    = np.random.default_rng(42)
    params = pnp.array(rng.uniform(-np.pi, np.pi, 6), requires_grad=True)
    opt    = qml.AdamOptimizer(stepsize=0.05)

    vqe_curve = []
    for step in range(N_VQE_STEPS):
        params, energy = opt.step_and_cost(vqe_circuit, params)
        vqe_curve.append(float(energy))

    E_vqe = vqe_curve[-1]
    print(f"VQE  ground-state energy : {E_vqe:+.6f} eV")
    print(f"Exact ground-state energy: {E_exact[0]:+.6f} eV")
    print(f"Error                    : {(E_vqe - E_exact[0])*1000:+.2f} meV")

    # Convergence plot
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(vqe_curve, lw=1.5, color="steelblue", label="VQE ⟨E⟩")
    ax.axhline(E_exact[0], color="tomato", ls="--", lw=1.5, label=f"Exact GS = {E_exact[0]:.4f} eV")
    ax.set_xlabel("Optimisation step", fontsize=11)
    ax.set_ylabel("Energy (eV)",        fontsize=11)
    ax.set_title("VQE convergence on 2-qubit TB Hamiltonian", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### VQE Result Summary

| | Value |
|---|---|
| Method | 2-qubit VQE, hardware-efficient ansatz (6 params) |
| Optimizer | Adam, 300 steps |
| VQE ground-state energy | ~4.30 eV |
| Exact NumPy eigenvalue | ~4.30 eV |
| Error | < 2 meV (< 0.05%) |

The VQE circuit exactly recovers the ground-state energy of the 4×4 projected TB Hamiltonian.  
For the full band dispersion along R→Γ run `src/python/vqe_dispersion.py` (saves `results/figures/vqe_dispersion.png`).

---

## 9. Phase 5 — Exciton Binding: Rytova-Keldysh and BSE

In 2D perovskites, the photo-excited electron and hole form a bound pair called an **exciton**.
The binding energy E_B determines whether excitons survive at room temperature (kT ≈ 26 meV).

Two effects make E_B large in 2D slabs:
1. **Geometric confinement** — the electron-hole pair is squeezed into a thin layer (0.6–3.1 nm)
2. **Dielectric contrast** — the high-ε inorganic slab (ε ≈ 6.1) is surrounded by low-ε organic spacers (ε ≈ 2.1), which barely screen the Coulomb attraction

### Rytova-Keldysh potential

The electron-hole interaction in a 2D slab follows the **Rytova-Keldysh (R-K)** potential:

$$V_{RK}(r) = -\frac{\pi e^2}{4\pi\varepsilon_0} \cdot \frac{1}{2\varepsilon_{\rm env}\, r_0} \left[ H_0\!\left(\frac{r}{r_0}\right) - Y_0\!\left(\frac{r}{r_0}\right) \right]$$

where $r_0 = \varepsilon_{\rm slab}\, d / (2\varepsilon_{\rm env})$ is the screening length.

### Two levels of calculation

| Level | Method | Key approximation |
|-------|--------|-------------------|
| 1 | Radial SE in real space | EMA: single effective mass μ from experiment |
| 2 | k-space BSE matrix | Full TB bands + intraband form factors |


In [ ]:
from rytova_keldysh import solve_binding_energy, r0_nm, EL_EXP, EL_BSE, V_RK_r

# Level-1: real-space radial SE
print('Level-1: Rytova-Keldysh binding energies (EMA)')
print(f"{'n':>3}  {'r0 (nm)':>8}  {'E_B L1 (meV)':>13}  {'Exp (meV)':>10}  {'Cho BSE':>9}")
print('-' * 50)
L1_results = {}
for n in range(1, 6):
    E_b, _ = solve_binding_energy(n)
    L1_results[n] = E_b
    cho = f"{EL_BSE[n]:>9}" if n in EL_BSE else f"{'—':>9}"
    print(f"{n:>3}  {r0_nm(n):>8.2f}  {E_b:>13.1f}  {EL_EXP[n]:>10}  {cho}")
print()
print('Trend correct: E_B decreases with n (thicker slab -> more screening)')
print('Level-1 ~45% below experiment (phonon renormalisation missing)')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rytova_keldysh import EPS_ENV, E2_OVER_4PIEPS0

r = np.linspace(0.05, 10, 500)
r0 = r0_nm(1)
V_rk = V_RK_r(r, r0)
V_coulomb = -E2_OVER_4PIEPS0 / (EPS_ENV * r)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(r, V_rk, label=f'R-K (n=1, r0={r0:.2f} nm)', lw=2)
ax.plot(r, V_coulomb, 'k--', lw=1.5, label='Bare Coulomb / eps_env')
ax.set_xlim(0, 10); ax.set_ylim(-3, 0.3)
ax.set_xlabel('r  (nm)'); ax.set_ylabel('V(r)  (eV)')
ax.set_title('Rytova-Keldysh vs Coulomb, n=1')
ax.legend(); plt.tight_layout(); plt.show()
print('V_RK is softer than 1/r at short r: slab screens close-range interaction')
print('V_RK -> Coulomb at large r: environment dominates long-range screening')


### BSE Level-2 results (pre-computed, Nk=41)

Running the full BSE at Nk=41 takes ~9 min. To recompute:
```bash
python3 src/python/bse_2d.py 41
python3 src/python/absorption_elliott.py 21
```


In [ ]:
BSE_L2 = {1: 160.1, 2: 88.4, 3: 66.4, 4: 48.6, 5: 39.1}  # meV (Nk=41)

print('Phase 5 exciton binding energies — summary')
print(f"{'n':>3}  {'Level-1 (meV)':>14}  {'BSE L2 (meV)':>13}  {'Exp (meV)':>10}  {'Cho BSE':>9}")
print('-' * 56)
for n in range(1, 6):
    cho = f"{EL_BSE[n]:>9}" if n in EL_BSE else f"{'—':>9}"
    print(f"{n:>3}  {L1_results[n]:>14.1f}  {BSE_L2[n]:>13.1f}  {EL_EXP[n]:>10}  {cho}")
print()
print('BSE L2 ~62% of Level-1: log-divergent 1/q kernel, k-grid cut at q_max=2 nm^-1')
print('Full convergence needs q_max~100 nm^-1 — achieved by Level-1 real-space grid')


In [ ]:
import os
from IPython.display import Image, display

fig_bse = os.path.join(REPO_ROOT, 'results', 'figures', 'bse_binding_energies.png')
fig_abs = os.path.join(REPO_ROOT, 'results', 'figures', 'absorption_bse_all.png')

if os.path.exists(fig_bse):
    print('BSE binding energy comparison:')
    display(Image(fig_bse))
else:
    print(f'Figure not found: {fig_bse} — run bse_2d.py 41')

if os.path.exists(fig_abs):
    print('Elliott absorption spectra n=1-5:')
    display(Image(fig_abs))
else:
    print(f'Figure not found: {fig_abs} — run absorption_elliott.py 21')


### Phase 5 Result Summary

| | Level-1 (R-K real space) | Level-2 (k-space BSE) |
|---|---|---|
| Method | Radial SE, EMA, finite differences | BSE matrix, TB bands + form factors |
| E_B (n=1) | 258 meV | 160 meV |
| E_B (n=5) | 109 meV | 39 meV |
| vs experiment | ~45% below | ~66% below |
| vs Cho BSE ref | ~15% below | ~47% below (k-grid truncation) |
| Key physics | R-K dielectric confinement | TB form factors, band tracking |
| Runtime | ~5 s | ~9 min (Nk=41) |

Remaining discrepancy vs experiment: phonon renormalisation (~30%) + finite k-grid in BSE.
